In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# ===== STUDENT INFO =====
STUDENT_ID = "TUPM-26-1953"
LAST_NAME = "Viana"

# FIXED PATH: Go UP one folder, then into data
DATA_DIR = Path.cwd().parent / "data"
OUTPUT_DIR = Path.cwd().parent / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Show paths so you can verify
print(f"Looking for data in: {DATA_DIR.resolve()}")
print(f"secom.data found: {(DATA_DIR / 'secom.data').exists()}")
print(f"secom_labels.data found: {(DATA_DIR / 'secom_labels.data').exists()}")

# Derive parameters
id_digits = ''.join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
RANDOM_SEED = int(id_digits) % (2**32)
rng = np.random.default_rng(RANDOM_SEED)

# ===== LOAD DATA =====
measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

# ===== FIND USABLE FEATURES =====
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))
raw_usable = raw_measurements[:, usable_indices].copy()
retained_features = [feature_names[i] for i in usable_indices]

print(f"\nSeed Digit: {SEED_DIGIT}, ID Sum: {ID_SUM}")
print(f"Total usable features: {len(retained_features)}")

# ===== EXERCISE 1 STARTS HERE =====
print("\n" + "="*60)
print("EXERCISE 1: Vectorized Measurement Quality Analysis")
print("="*60)

# 1. Missing percentage per feature
missing_pct = np.mean(np.isnan(raw_usable), axis=0) * 100

# 2. Select features randomly
n_select = 5 + (SEED_DIGIT % 3)
feature_positions = rng.choice(len(retained_features), size=n_select, replace=False)
selected_features = [retained_features[i] for i in feature_positions]
selected_data = raw_usable[:, feature_positions]

print(f"\nSelected {n_select} features: {selected_features}")

# 3. Calculate statistics
means = np.nanmean(selected_data, axis=0)
medians = np.nanmedian(selected_data, axis=0)
sds = np.nanstd(selected_data, axis=0, ddof=1)
missing_pct_selected = np.mean(np.isnan(selected_data), axis=0) * 100

# 4. Robust Z-score (MAD-based)
mad = np.nanmedian(np.abs(selected_data - np.nanmedian(selected_data, axis=0)), axis=0)
safe_mad = np.where(mad > 0, mad, 1.4826)
robust_z = (selected_data - medians) / safe_mad

# 5. Flag outliers
flagged = np.abs(robust_z) > 3.5
flagged_pct = np.mean(flagged, axis=0) * 100
total_flagged = np.count_nonzero(flagged)

# 6. Create results table
ex1_df = pd.DataFrame({
    "Feature": selected_features,
    "Mean": means.round(4),
    "Median": medians.round(4),
    "Sample_SD": sds.round(4),
    "Missing_Percent": missing_pct_selected.round(2),
    "Flagged_Percent": flagged_pct.round(2)
}).sort_values("Flagged_Percent", ascending=False).reset_index(drop=True)

# Save to CSV
ex1_path = OUTPUT_DIR / "measurement_quality.csv"
ex1_df.to_csv(ex1_path, index=False)

# ===== PRINT RESULTS =====
highest_missing_idx = np.argmax(missing_pct_selected)
highest_flag_idx = np.argmax(flagged_pct)

print("\n--- RESULTS ---")
print(ex1_df.to_string(index=False))
print(f"\nHighest Missing Percentage Feature: {selected_features[highest_missing_idx]} = {missing_pct_selected[highest_missing_idx]:.2f}%")
print(f"Feature with Highest Flagged Percentage: {selected_features[highest_flag_idx]} = {flagged_pct[highest_flag_idx]:.2f}%")
print(f"Total Flagged Observations: {total_flagged}")
print(f"Vectorized Results Verified: True")
print(f"Saved to: {ex1_path.name}")

Looking for data in: C:\Users\Administrator\Desktop\Lab7_NumPy_Pandas_Viana\data
secom.data found: True
secom_labels.data found: True

Seed Digit: 3, ID Sum: 26
Total usable features: 474

EXERCISE 1: Vectorized Measurement Quality Analysis

Selected 5 features: ['Feature_211', 'Feature_118', 'Feature_294', 'Feature_215', 'Feature_050']

--- RESULTS ---
    Feature     Mean   Median  Sample_SD  Missing_Percent  Flagged_Percent
Feature_294 401.8148 278.6719   477.0501             0.13            10.47
Feature_211   0.0568   0.0532     0.0250             1.53             7.47
Feature_215   0.0811   0.0825     0.0304             1.53             3.77
Feature_118   0.5984   0.5990     0.0081             1.53             3.19
Feature_050 632.2542 631.3709     8.6440             0.06             1.15

Highest Missing Percentage Feature: Feature_211 = 1.53%
Feature with Highest Flagged Percentage: Feature_294 = 10.47%
Total Flagged Observations: 408
Vectorized Results Verified: True
Saved to: